In [1]:
import os
import pandas as pd
import numpy as np

os.chdir(r"D:\Projects\berlin-electricity-temperature")

# Load the raw temperature file we downloaded earlier
temp_raw = pd.read_csv("data/raw/dwd_temperature/berlin_tempelhof_raw.csv")

print("Shape:", temp_raw.shape)
print("Columns:", list(temp_raw.columns))
print(temp_raw.head(3))

Shape: (570247, 6)
Columns: ['STATIONS_ID', 'MESS_DATUM', 'QN_9', 'TT_TU', 'RF_TU', 'eor']
   STATIONS_ID  MESS_DATUM  QN_9  TT_TU  RF_TU  eor
0          433  1951010101     5  -10.7   88.0  eor
1          433  1951010102     5  -11.0   87.0  eor
2          433  1951010103     5  -11.2   87.0  eor


In [2]:
# Keep only temperature and humidity columns
temp = temp_raw[['MESS_DATUM', 'TT_TU', 'RF_TU']].copy()

# Rename to simple names
temp.columns = ['datetime', 'temperature_c', 'humidity_pct']

print("Shape:", temp.shape)
print(temp.head(3))

Shape: (570247, 3)
     datetime  temperature_c  humidity_pct
0  1951010101          -10.7          88.0
1  1951010102          -11.0          87.0
2  1951010103          -11.2          87.0


In [5]:
# DWD uses -999 to represent missing temperature readings
# We replace -999 with NaN (pandas standard for missing)

temp['temperature_c'] = temp['temperature_c'].replace(-999, np.nan)
temp['humidity_pct'] = temp['humidity_pct'].replace(-999, np.nan)

# Check how many missing values we have
missing_temp = temp['temperature_c'].isna().sum()
missing_hum  = temp['humidity_pct'].isna().sum()
total        = len(temp)

print(f"Missing temperature : {missing_temp} ({missing_temp/total*100:.2f}%)")
print(f"Missing humidity    : {missing_hum} ({missing_hum/total*100:.2f}%)")

Missing temperature : 342 (0.06%)
Missing humidity    : 691 (0.12%)


In [7]:
import os
import pandas as pd
import numpy as np

os.chdir(r"D:\Projects\berlin-electricity-temperature")

# Load raw file
temp_raw = pd.read_csv("data/raw/dwd_temperature/berlin_tempelhof_raw.csv")

# Keep only what we need
temp = temp_raw[['MESS_DATUM', 'TT_TU', 'RF_TU']].copy()
temp.columns = ['datetime', 'temperature_c', 'humidity_pct']

# Fix date format
temp['datetime'] = pd.to_datetime(temp['datetime'], format='%Y%m%d%H')

# Replace DWD missing value code with NaN
temp['temperature_c'] = temp['temperature_c'].replace(-999, np.nan)
temp['humidity_pct']  = temp['humidity_pct'].replace(-999, np.nan)

# Fill small gaps using interpolation
temp['temperature_c'] = temp['temperature_c'].interpolate(method='linear', limit=3)
temp['humidity_pct']  = temp['humidity_pct'].interpolate(method='linear', limit=3)

# Filter to study period
temp = temp[temp['datetime'] >= '2015-01-01'].reset_index(drop=True)

# Final check
print("Shape:", temp.shape)
print("First date:", temp['datetime'].iloc[0])
print("Last date:", temp['datetime'].iloc[-1])
print("Missing temperature:", temp['temperature_c'].isna().sum())
print("Missing humidity:", temp['humidity_pct'].isna().sum())
print(temp.head(3))

Shape: (97732, 3)
First date: 2015-01-01 00:00:00
Last date: 2026-02-26 23:00:00
Missing temperature: 301
Missing humidity: 541
             datetime  temperature_c  humidity_pct
0 2015-01-01 00:00:00            5.2          95.0
1 2015-01-01 01:00:00            5.1          95.0
2 2015-01-01 02:00:00            4.8          94.0


In [10]:
# Load the saved energy data
energy = pd.read_csv("data/processed/energy_clean.csv")
energy['datetime'] = pd.to_datetime(energy['datetime'])

print("Energy loaded:", energy.shape)
print(energy.head(2))

Energy loaded: (97789, 6)
             datetime demand_mwh wind_offshore_mwh wind_onshore_mwh solar_mwh  \
0 2015-01-01 00:00:00  44,600.25            516.50         8,128.00      0.00   
1 2015-01-01 01:00:00  43,454.75            516.25         8,297.50      0.00   

    gas_mwh  
0  1,226.25  
1    870.75  


In [11]:
# Now merge temperature with energy
master = pd.merge(energy, temp, on='datetime', how='inner')

print("Shape:", master.shape)
print("Columns:", list(master.columns))
print(master.head(3))

Shape: (97721, 8)
Columns: ['datetime', 'demand_mwh', 'wind_offshore_mwh', 'wind_onshore_mwh', 'solar_mwh', 'gas_mwh', 'temperature_c', 'humidity_pct']
             datetime demand_mwh wind_offshore_mwh wind_onshore_mwh solar_mwh  \
0 2015-01-01 00:00:00  44,600.25            516.50         8,128.00      0.00   
1 2015-01-01 01:00:00  43,454.75            516.25         8,297.50      0.00   
2 2015-01-01 02:00:00  41,963.25            514.00         8,540.00      0.00   

    gas_mwh  temperature_c  humidity_pct  
0  1,226.25            5.2          95.0  
1    870.75            5.1          95.0  
2    809.50            4.8          94.0  


In [14]:
# These columns need to be converted from text to numbers
number_cols = ['demand_mwh', 'wind_offshore_mwh', 'wind_onshore_mwh', 'solar_mwh', 'gas_mwh']

for col in number_cols:
    # Remove commas from numbers like "44,600.25" → "44600.25"
    master[col] = master[col].astype(str).str.replace(',', '').astype(float)

print("Columns converted to numbers!")
print(master[number_cols].dtypes)
print(master[number_cols].head(3))

Columns converted to numbers!
demand_mwh           float64
wind_offshore_mwh    float64
wind_onshore_mwh     float64
solar_mwh            float64
gas_mwh              float64
dtype: object
   demand_mwh  wind_offshore_mwh  wind_onshore_mwh  solar_mwh  gas_mwh
0    44600.25             516.50            8128.0        0.0  1226.25
1    43454.75             516.25            8297.5        0.0   870.75
2    41963.25             514.00            8540.0        0.0   809.50


In [15]:
# Total wind = offshore + onshore combined
master['total_wind_mwh'] = master['wind_offshore_mwh'] + master['wind_onshore_mwh']

# Total renewables = wind + solar
master['total_renewable_mwh'] = master['total_wind_mwh'] + master['solar_mwh']

# Renewable share = what % of demand is covered by renewables
master['renewable_share_pct'] = (master['total_renewable_mwh'] / master['demand_mwh']) * 100

# Energy gap = how much demand renewables CANNOT cover
master['energy_gap_mwh'] = master['demand_mwh'] - master['total_renewable_mwh']

# Time columns
master['year']        = master['datetime'].dt.year
master['month']       = master['datetime'].dt.month
master['hour']        = master['datetime'].dt.hour
master['day_of_week'] = master['datetime'].dt.dayofweek
master['season']      = master['month'].map({
    12:'Winter', 1:'Winter', 2:'Winter',
    3:'Spring',  4:'Spring',  5:'Spring',
    6:'Summer',  7:'Summer',  8:'Summer',
    9:'Autumn',  10:'Autumn', 11:'Autumn'
})

# Heating/Cooling Degree Days (industry standard metric)
BASE_TEMP = 15  # °C baseline for Germany
master['HDD'] = (BASE_TEMP - master['temperature_c']).clip(lower=0)  # heating needed
master['CDD'] = (master['temperature_c'] - BASE_TEMP).clip(lower=0)  # cooling needed

print("New columns added!")
print("Columns now:", list(master.columns))
print(master.head(3))

New columns added!
Columns now: ['datetime', 'demand_mwh', 'wind_offshore_mwh', 'wind_onshore_mwh', 'solar_mwh', 'gas_mwh', 'temperature_c', 'humidity_pct', 'total_wind_mwh', 'total_renewable_mwh', 'renewable_share_pct', 'energy_gap_mwh', 'year', 'month', 'hour', 'day_of_week', 'season', 'HDD', 'CDD']
             datetime  demand_mwh  wind_offshore_mwh  wind_onshore_mwh  \
0 2015-01-01 00:00:00    44600.25             516.50            8128.0   
1 2015-01-01 01:00:00    43454.75             516.25            8297.5   
2 2015-01-01 02:00:00    41963.25             514.00            8540.0   

   solar_mwh  gas_mwh  temperature_c  humidity_pct  total_wind_mwh  \
0        0.0  1226.25            5.2          95.0         8644.50   
1        0.0   870.75            5.1          95.0         8813.75   
2        0.0   809.50            4.8          94.0         9054.00   

   total_renewable_mwh  renewable_share_pct  energy_gap_mwh  year  month  \
0              8644.50            19.382178

In [16]:
# Save to processed folder — this is our foundation for everything
master.to_csv("data/processed/master_dataset.csv", index=False)

print("✅ Master dataset saved!")
print("Shape:", master.shape)
print("File saved to: data/processed/master_dataset.csv")

✅ Master dataset saved!
Shape: (97721, 19)
File saved to: data/processed/master_dataset.csv
